# Game loop: wiring vision, arm, engine and teacher together

**Deliverable:** the complete game loop, runnable without hardware. Everything is injected:

| Dependency | Real robot | Gazebo | Software (this notebook) |
|---|---|---|---|
| vision | `gambit_vision` board reader | camera topic + vision | `FakeVision` (reads the simulated board) |
| arm | `gambit_arm` primitives | `gambit_sim` teleport | `FakeArm` (edits the simulated board) |
| robot policy | teacher (`gambit_teach`) | same | `EnginePolicy(MaterialEngine())` |
| human | a person | a script | `SimulatedHuman` (any engine) |

`SimulatedGame` plays engine vs random entirely in software: the simulated human moves pieces on a
shared "physical" piece map, the fake vision reads that map, the game manager detects the move,
the robot plans arm steps, the fake arm executes them, and the camera verifies. Logs are written
with `logger.ipynb`, think times collected with `hesitation.ipynb`.

**TODO checklist**
- [ ] Fakes: `PhysicalBoard`, `FakeArm`, `FakeVision`, `SimulatedHuman`.
- [ ] `make_log_listener` (MoveEvent -> MoveRecord, optional analyst evals).
- [ ] `SimulatedGame` (runs in the self-check once everything above works).
- [ ] Fault injection: `FakeArm(fail_rate=...)`, `FakeVision(noise=...)` to exercise recovery once
      `recovery.choose_recovery_action` exists.
- [ ] Real wiring in `run_play_game.ipynb` (vision node / arm driver adapters).

In [ ]:
%run ./game_manager.ipynb
%run ./logger.ipynb
%run ./hesitation.ipynb

In [ ]:
import random
import tempfile
from collections import Counter
from pathlib import Path
from typing import Optional

import chess

## The simulated physical world

`PhysicalBoard` is the ground truth the fake camera sees: a `{square: Piece}` map plus a graveyard
counter. The fake arm and the simulated human both change it; the game manager only ever sees it
through `FakeVision`.

In [ ]:
class PhysicalBoard:
    def __init__(self, fen: str = chess.STARTING_FEN, spare_queens: int = SPARE_QUEENS):
        """`pieces`: {square: Piece} from the FEN; `graveyard`: Counter starting with the spare queens."""
        # TODO: implement
        raise NotImplementedError

    def observation(self, t: float = 0.0) -> BoardObservation:
        """A perfect camera reading of the current pieces at time t (all confidences 1.0)."""
        # Design: build a placement string from the piece map (chess.BaseBoard.empty + set_piece_at).
        # TODO: implement
        raise NotImplementedError


class FakeArm:
    """ArmInterface on a PhysicalBoard. `fail_rate`: chance a primitive does nothing and returns False."""

    def __init__(self, world: PhysicalBoard, fail_rate: float = 0.0, seed: int = 0):
        """Store the world and fail rate; seeded RNG; `held` piece (None); `calls` log of primitives."""
        # TODO: implement
        raise NotImplementedError

    def _fail(self) -> bool:
        """Random failure with probability fail_rate."""
        # TODO: implement
        raise NotImplementedError

    def pick(self, square):
        """Lift the piece on `square` into the gripper. False if gripper busy, square empty, or fail."""
        # Design: record the call in `calls` (tests inspect it); behave like a real arm would refuse.
        # TODO: implement
        raise NotImplementedError

    def place(self, square):
        """Put the held piece on `square`. False if nothing held, square occupied, or fail."""
        # TODO: implement
        raise NotImplementedError

    def move_piece(self, from_sq, to_sq):
        """pick then place."""
        # TODO: implement
        raise NotImplementedError

    def capture_to_graveyard(self, square):
        """Pick the piece and drop it into the world's graveyard counter."""
        # TODO: implement
        raise NotImplementedError

    def place_from_graveyard(self, piece, square):
        """Take `piece` out of the graveyard (False if none) and place it on `square`."""
        # TODO: implement
        raise NotImplementedError

    def park(self):
        """Nothing to do in software; report success."""
        # TODO: implement
        raise NotImplementedError


class FakeVision:
    """Reads the PhysicalBoard. `before_read` lets the simulated human act between frames."""

    def __init__(self, world: PhysicalBoard, clock=None):
        self.world, self.before_read = world, None
        self.clock = clock or (lambda: 0.0)

    def read(self) -> BoardObservation:
        """Call the before_read hook (if set), then observe the world at clock()."""
        # TODO: implement
        raise NotImplementedError


class SimulatedHuman:
    """Moves pieces on the PhysicalBoard when it is the human's turn on the manager's rules board."""

    def __init__(self, engine: EngineLike, world: PhysicalBoard, color: chess.Color):
        self.engine, self.world, self.color = engine, world, color
        self._moved_at_ply = None

    def maybe_move(self, rules_board: chess.Board) -> None:
        """If it is our turn (and we have not moved at this ply yet), physically make a move."""
        # Design:
        # - Guard: our colour to move, game not over, not already moved at this ply (vision reads
        #   many frames per turn; the human must move once).
        # - Pick a move with the engine, turn it into steps with plan_physical_move against the
        #   world's graveyard, replay them with apply_steps on the world's pieces.
        # - A real human brings a missing promotion piece from the box: on MissingPieceError fall
        #   back to promoting to a queen (or just conjure the piece) - decide.
        # TODO: implement
        raise NotImplementedError

## MoveEvent -> log record

`make_log_listener` turns game-manager events into `MoveRecord`s. If an `analyst` engine is given,
each position is analysed for the eval and best move (that is what the teaching side needs).

In [ ]:
def make_log_listener(logger: GameLogger, analyst: Optional[EngineLike] = None):
    """Return a listener fn(event: MoveEvent) that writes one MoveRecord per event."""
    # Design:
    # - Rebuild the boards before/after from fen_before + move.
    # - With an analyst: best move in the position before, eval of the position after. Engine
    #   scores are side-to-move POV; MoveRecord.eval_cp is White POV, so flip when Black is to
    #   move. Skip the eval when the game is over after the move.
    # - Copy think time, robot attempts, plan and extras from the event.
    # TODO: implement
    raise NotImplementedError

## SimulatedGame

In [ ]:
class _TickClock:
    """Deterministic clock: each call advances 0.1 s (so think times are non-zero and reproducible)."""

    def __init__(self):
        self.t = 0.0

    def __call__(self):
        # TODO: advance and return the time
        raise NotImplementedError


class SimulatedGame:
    def __init__(self, robot_policy: MovePolicy, human_engine: EngineLike,
                 human_color: chess.Color = chess.WHITE, fen: str = chess.STARTING_FEN,
                 log_dir: Optional[Path] = None, analyst: Optional[EngineLike] = None,
                 arm_fail_rate: float = 0.0):
        """Build the whole software world and a GameManager on top of it."""
        # Design:
        # - One shared _TickClock for vision and manager; PhysicalBoard(fen), FakeArm, FakeVision,
        #   SimulatedHuman, a HesitationTracker listener.
        # - If log_dir: a GameLogger with White/Black headers and a log listener (make_log_listener).
        # - GameManager(vision, arm, robot_policy, human_color, board=chess.Board(fen), listeners, clock).
        # - Hook vision.before_read to _human_turn so the human acts between camera frames.
        # TODO: implement
        raise NotImplementedError

    def _human_turn(self) -> None:
        """The human only moves while the manager is in WAIT_HUMAN."""
        # TODO: implement
        raise NotImplementedError

    def play(self, max_plies: Optional[int] = None) -> chess.Board:
        """Run the manager; finish the log with the result (or '*' if unfinished)."""
        # TODO: implement
        raise NotImplementedError

## Self-check: a few plies of MaterialEngine (robot) vs RandomEngine (human), all in software

In [ ]:
# Self-check plan (write these as asserts once the functions above work):
# - SimulatedGame(EnginePolicy(MaterialEngine(depth=1)), RandomEngine(seed=3), WHITE, log_dir=tmp,
#   analyst=MaterialEngine(depth=1)).play(max_plies=8):
#   * board.ply() == 8; the physical board's placement equals the rules board;
#   * VERIFY appears in the manager trace and RECOVER does not;
#   * read_jsonl: by == human, robot alternating 4 times, every eval present;
#   * the PGN's mainline equals board.move_stack; the tracker has 4 human feature rows.
# - Robot promotion: robot white, FEN "8/P6k/8/8/8/8/8/K7 w - - 0 1", 1 ply -> queen promotion,
#   one spare white queen used, white pawn in the graveyard, "graveyard Q -> a8" in arm.calls.
# Example shape of a test:
# with tempfile.TemporaryDirectory() as tmp:
#     sim = SimulatedGame(EnginePolicy(MaterialEngine(depth=1)), RandomEngine(seed=3), log_dir=Path(tmp))
#     assert sim.play(max_plies=8).ply() == 8
print("Self-check: not written yet (see plan above)")